# California Housing Data Preprocessing

Inspect the data and prepare it for a machine-learning model.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("housing.csv")
df.head()

## Preprocessing functions

Run this cell to define the functions, then use the cells below to run the analysis and preprocessing.

In [1]:
def information(df):
    print("DataFrame information:")
    df.info()
    print("\nSummary statistics:")
    print(df.describe())


def missing_values(df):
    missing = df.isnull().sum()
    return missing[missing > 0].sort_values(ascending=False)


def quartis(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return q1, q3, q3 - q1


def boundaries(series):
    q1, q3, iqr = quartis(series)
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def analysis_On_data(df):
    information(df)
    print("\nMissing values:")
    print(missing_values(df))

    low, high = boundaries(df["median_house_value"])
    outliers = df["median_house_value"].lt(low) | df["median_house_value"].gt(high)
    print("\nHouse value outlier boundaries:")
    print("Lower:", low)
    print("Upper:", high)
    print("Number of outliers:", outliers.sum())

    print("\nOcean proximity counts:")
    print(df["ocean_proximity"].value_counts())


def preprocess_data(df):
    data = df.copy()
    data["total_bedrooms"] = data["total_bedrooms"].fillna(data["total_bedrooms"].median())
    data = pd.get_dummies(data, columns=["ocean_proximity"], drop_first=True, dtype=int)

    X = data.drop(columns=["median_house_value"])
    y = data["median_house_value"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("Before preprocessing (first 5 training rows):")
    print(df.loc[X_train.index].drop(columns=["median_house_value"]).head())

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\nAfter preprocessing (same rows):")
    print(pd.DataFrame(
        X_train_scaled, columns=X_train.columns, index=X_train.index
    ).head().round(2))

    return X_train_scaled, X_test_scaled, y_train, y_test

## Run the data analysis

In [ ]:
analysis_On_data(df)

## Run preprocessing

Fill missing bedroom values, encode the ocean category, split into training and test sets, and scale the features. The two previews compare the same training rows before and after scaling.

In [ ]:
X_train_scaled, X_test_scaled, y_train, y_test = preprocess_data(df)
print("\nTraining features:", X_train_scaled.shape)
print("Test features:", X_test_scaled.shape)
print("Training targets:", y_train.shape)
print("Test targets:", y_test.shape)